# Analysis of coexpressed gene pairs
## Setup for tissue specific GO
### Author: Martin Loza
### Date: 26/01/19


In [17]:
# Change R language to English
Sys.setenv(LANGUAGE = "en")

# Init
suppressPackageStartupMessages({
    library(dplyr)
    library(stringr)
    library(ggplot2)
    library(patchwork)
    library(TissueEnrich)
    library(ggrepel)
})

# Local variables 
seed = 777
date = "260119"

# Define colors for strand plots
red = "#E41A1C"
blue = "#377EB8"
# Define colors for gene types
green = "#4DAF4A"
purple = "#984EA3"
gray = "gray50"
text_size = 18
width = 18.6
height = 5
dot_size = 4
line_size = 1.5
dpi = 300

encode_results_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENCODE/coexpression_results/ENCODE_coexpression_results_selected_lncRNA_TF_pairs_251216.tsv"
encode_data_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/ENCODE/selected_gene_pairs/log_normalized_tpm_selected_lncRNA_TF_genes_251216.tsv"
gtex_results_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/GTEx/coexpression_results/GTEx_coexpression_results_selected_lncRNA_TF_pairs_251216.tsv"
gtex_data_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/GTEx/selected_gene_pairs/log_normalized_tpm_selected_lncRNA_TF_genes_251218.tsv"
# gtex_data_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/GTEx/normalized/log_normalized_tpm_251218.tsv"
out_dir = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/09_Figure_Correlation_ENCODE_GTEx/Results/"
# Local Functions



### Load and setup the data

In [18]:
# Load the ENCODE coexpression results
encode_results <- read.table(encode_results_file, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
cat("Number of gene pairs: ", nrow(encode_results), "\n")
head(encode_results)

Number of gene pairs:  777 


,gene_pair_id,lncRNA_gene_id,TF_gene_id,pearson_correlation,pearson_pvalue,pearson_fdr,spearman_correlation,spearman_pvalue,spearman_fdr
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000099869_ENSG00000284779,ENSG00000099869,ENSG00000284779,0.15254446,0.520842118,0.622606655,0.79558726,2.742561e-05,0.0002767494
2,ENSG00000124915_ENSG00000124920,ENSG00000124915,ENSG00000124920,0.02603510,0.913239471,0.942346705,0.09921048,6.773049e-01,0.7496665890
3,ENSG00000163081_ENSG00000135903,ENSG00000163081,ENSG00000135903,0.08711872,0.714950344,0.789862789,0.14909580,5.304196e-01,0.6253961227
4,ENSG00000163597_ENSG00000284526,ENSG00000163597,ENSG00000284526,0.45853176,0.042012303,0.076092214,0.51684030,1.962607e-02,0.0460801246
5,ENSG00000164621_ENSG00000113658,ENSG00000164621,ENSG00000113658,0.62966915,0.002928728,0.007585405,0.64918355,1.953723e-03,0.0078655079
6,ENSG00000165511_ENSG00000165512,ENSG00000165511,ENSG00000165512,0.63206409,0.002790792,0.007294835,0.48156522,3.156428e-02,0.0666341940


In [19]:
# Load the GTEx coexpression results
gtex_results <- read.table(gtex_results_file, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
cat("Number of gene pairs: ", nrow(gtex_results), "\n")
head(gtex_results)

Number of gene pairs:  930 


,gene_pair_id,lncRNA_gene_id,TF_gene_id,pearson_correlation,pearson_pvalue,pearson_fdr,spearman_correlation,spearman_pvalue,spearman_fdr
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000099869_ENSG00000284779,ENSG00000099869,ENSG00000284779,0.6941562,5.240802e-11,1.275902e-10,0.1086102,3.779769e-01,3.989994e-01
2,ENSG00000124915_ENSG00000124920,ENSG00000124915,ENSG00000124920,0.1554977,2.054372e-01,2.299116e-01,0.8703644,5.524228e-22,3.696066e-21
3,ENSG00000163081_ENSG00000135903,ENSG00000163081,ENSG00000135903,0.8500990,4.760093e-20,2.354727e-19,0.6647782,6.321785e-10,1.373659e-09
4,ENSG00000163597_ENSG00000284526,ENSG00000163597,ENSG00000284526,0.3792864,1.423800e-03,1.924613e-03,0.5249838,4.305883e-06,6.663013e-06
5,ENSG00000164621_ENSG00000113658,ENSG00000164621,ENSG00000113658,0.1412541,2.505610e-01,2.764196e-01,0.1524956,2.144311e-01,2.318848e-01
6,ENSG00000165511_ENSG00000165512,ENSG00000165511,ENSG00000165512,0.5629531,5.819021e-07,1.040710e-06,0.6593503,9.717994e-10,2.077640e-09


Load the expression data of selected gene pairs

In [20]:
# Load the GTEx expression data
gtex_data <- read.table(gtex_data_file, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
# Change the first column name to "gene_id"
colnames(gtex_data)[1] <- "gene_id"
cat("GTEx expression data dimensions: ", dim(gtex_data), "\n")
head(gtex_data[, 1:5])

GTEx expression data dimensions:  2058 69 


,gene_id,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000272438,0.06808566,0.1234452,0.27456568,0.2352768
2,ENSG00000223764,0.46315775,0.9997234,1.60198420,1.9059241
3,ENSG00000187634,0.39083866,1.3746670,1.58750105,1.9401048
4,ENSG00000272512,1.82029833,1.5857946,0.49259098,2.2775975
5,ENSG00000188290,3.16147126,3.2602481,1.38873139,3.8918407
6,ENSG00000197921,0.23182734,0.3240786,0.02956657,0.3727732


Load the gene_pairs original information to recover interesting gene patterns

In [21]:
gene_pairs_file = "/mnt/c/Users/Marti/Documents/Projects/lncRNA_TF_pairs_analysis/Data/Annotated_ncRNA_PCG_pairs/unique/human_unique_lncRNA_TF_pairs_10000bp_251215.tsv"
gene_pairs <- read.table(gene_pairs_file, header = TRUE, sep = "\t", stringsAsFactors = FALSE)
cat("Number of gene pairs: ", nrow(gene_pairs), "\n")
head(gene_pairs)

Number of gene pairs:  1978 


,chromosome,ncRNA_id,ncrna_tss,ncrna_gene_name,ncrna_strand,gene_biotype,pcg_id,pcg_gene_name,pcg_tss,dna_distance,⋯,Family,is_TF,abs_strand_distance,ncrna_gene_id,pcg_gene_id,gene_pair_id,gene_name_pair_id,ncrna_is_mane,tf_is_mane,mane_priority
,<chr>,<chr>,<int>,<chr>,<int>,<chr>,<chr>,<chr>,<int>,<int>,⋯,<chr>,<lgl>,<int>,<chr>,<chr>,<chr>,<chr>,<lgl>,<lgl>,<int>
1,11,ENST00000381363,2140644,IGF2-AS,1,lncRNA,ENST00000643349,unnamed,2149603,8959,⋯,ZBTB,TRUE,8959,ENSG00000099869,ENSG00000284779,ENSG00000099869_ENSG00000284779,IGF2-AS_unnamed,FALSE,FALSE,4
2,11,ENST00000833483,61756482,MYRF-AS1,-1,lncRNA,ENST00000265460,MYRF,61755389,-1093,⋯,NDT80_PhoG,TRUE,1093,ENSG00000124915,ENSG00000124920,ENSG00000124915_ENSG00000124920,MYRF-AS1_MYRF,FALSE,FALSE,4
3,2,ENST00000440903,222302978,CCDC140,1,lncRNA,ENST00000392070,PAX3,222298998,-3980,⋯,PAX,TRUE,3980,ENSG00000163081,ENSG00000135903,ENSG00000163081_ENSG00000135903,CCDC140_PAX3,FALSE,TRUE,2
4,17,ENST00000738117,76562607,SNHG16,1,lncRNA,ENST00000713548,unnamed,76569648,7041,⋯,ZBTB,TRUE,7041,ENSG00000163597,ENSG00000284526,ENSG00000163597_ENSG00000284526,SNHG16_unnamed,FALSE,TRUE,2
5,5,ENST00000809704,136132805,SMAD5-AS1,-1,lncRNA,ENST00000545279,SMAD5,136132845,40,⋯,MH1,TRUE,40,ENSG00000164621,ENSG00000113658,ENSG00000164621_ENSG00000113658,SMAD5-AS1_SMAD5,FALSE,TRUE,2
6,10,ENST00000625168,45000920,ZNF22-AS1,-1,lncRNA,ENST00000298299,ZNF22,45000923,3,⋯,zf-C2H2,TRUE,3,ENSG00000165511,ENSG00000165512,ENSG00000165511_ENSG00000165512,ZNF22-AS1_ZNF22,FALSE,TRUE,2


### Analysis

Let's select significant gene pairs within each dataset

In [22]:
# Select significant pairs based on adjusted p-value
adjusted_pvalue_threshold <- 0.05
selected_encode <- encode_results %>%
    filter(pearson_fdr <= adjusted_pvalue_threshold & spearman_fdr <= adjusted_pvalue_threshold)

selected_gtex <- gtex_results %>%
    filter(pearson_fdr <= adjusted_pvalue_threshold & spearman_fdr <= adjusted_pvalue_threshold)

cat("Number of significant gene pairs in ENCODE: ", nrow(selected_encode), "\n")
cat("Number of significant gene pairs in GTEx: ", nrow(selected_gtex), "\n")

Number of significant gene pairs in ENCODE:  295 


Number of significant gene pairs in GTEx:  751 


Now, let's select the significant pairs across datasets

In [23]:
# Get the intersection of gene pairs
common_pairs <- intersect(selected_encode$gene_pair_id, selected_gtex$gene_pair_id)
cat("Number of common significant gene pairs: ", length(common_pairs), "\n")

Number of common significant gene pairs:  260 


Let's save them for GO analysis

In [24]:
common_pairs_df <- selected_encode %>%
    filter(gene_pair_id %in% common_pairs) 
head(common_pairs_df)

,gene_pair_id,lncRNA_gene_id,TF_gene_id,pearson_correlation,pearson_pvalue,pearson_fdr,spearman_correlation,spearman_pvalue,spearman_fdr
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000166770_ENSG00000198046,ENSG00000166770,ENSG00000198046,0.7384413,2.008462e-04,7.962118e-04,0.7301468,2.572352e-04,1.580054e-03
2,ENSG00000167912_ENSG00000198846,ENSG00000167912,ENSG00000198846,0.8548898,1.580584e-06,1.292752e-05,0.7996247,2.328468e-05,2.444891e-04
3,ENSG00000176593_ENSG00000166704,ENSG00000176593,ENSG00000166704,0.8665018,7.799795e-07,6.659825e-06,0.8337098,4.965214e-06,6.768371e-05
4,ENSG00000177133_ENSG00000142611,ENSG00000177133,ENSG00000142611,0.8947618,1.020600e-07,1.166185e-06,0.5982659,5.328015e-03,1.649350e-02
5,ENSG00000177738_ENSG00000172262,ENSG00000177738,ENSG00000172262,0.5705344,8.616182e-03,1.951829e-02,0.6255639,3.178355e-03,1.142145e-02
6,ENSG00000181097_ENSG00000181135,ENSG00000181097,ENSG00000181135,0.5882138,6.372868e-03,1.514287e-02,0.5872897,6.476816e-03,1.943045e-02


### Annotate TF genes with tissue specificity using TissueEnrich

In [25]:
# Extract unique TF gene IDs
tf_genes <- unique(common_pairs_df$TF_gene_id)
cat("Number of unique TF genes: ", length(tf_genes), "\n")
head(tf_genes)

Number of unique TF genes:  231 


[1] "ENSG00000198046" "ENSG00000198846" "ENSG00000166704" "ENSG00000142611"
[5] "ENSG00000172262" "ENSG00000181135"

In [33]:
# convert GTEx data to summarized experiment object
# We want to use the selected genes only
gtex_data <- gtex_data %>% filter(!duplicated(gene_id))
rownames(gtex_data) <- gtex_data$gene_id
sel_gtex_data <- gtex_data[tf_genes, -1]
gtex_se <- SummarizedExperiment(assays = list(counts = as.matrix(sel_gtex_data)), 
                              rowData = rownames(sel_gtex_data), colData = colnames(sel_gtex_data))
cat("GTEx summarized experiment dimensions: ", dim(gtex_se), "\n")
gtex_se

GTEx summarized experiment dimensions:  231 68 


class: SummarizedExperiment 
dim: 231 68 
metadata(0):
assays(1): counts
rownames(231): ENSG00000198046 ENSG00000198846 ... ENSG00000133937
  ENSG00000140262
rowData names(1): X
colnames(68): Adipose_Subcutaneous Adipose_Visceral_Omentum ... Vagina
  Whole_Blood
colData names(1): X

In [43]:
output <- teGeneRetrieval(gtex_se)
output_df <- assay(output) %>% as.data.frame()
head(output_df)

,Gene,Tissue,Group
,<chr>,<chr>,<chr>
1,ENSG00000198046,All,Mixed
2,ENSG00000198846,All,Mixed
3,ENSG00000166704,All,Mixed
4,ENSG00000142611,All,Mixed
5,ENSG00000172262,All,Expressed-In-All
6,ENSG00000181135,All,Mixed


In [51]:
# Let's get tissue specific genes
selected_genes <- output_df %>%
    filter(Tissue != "All")
table(selected_genes$Tissue) %>% sort(decreasing = TRUE) %>% head(10)




                               Testis                      Brain_Cerebellum 
                                   10                                     8 
          Brain_Cerebellar_Hemisphere       Skin_Not_Sun_Exposed_Suprapubic 
                                    7                                     7 
           Skin_Sun_Exposed_Lower_leg           Brain_Caudate_basal_ganglia 
                                    7                                     6 
                      Pancreas_Islets                    Brain_Hypothalamus 
                                    6                                     5 
Brain_Nucleus_accumbens_basal_ganglia           Brain_Putamen_basal_ganglia 
                                    5                                     5 

Let's select the top 5 tissues 

In [52]:
top_tissues <- selected_genes %>% 
    group_by(Tissue) %>%
    summarise(Count = n()) %>%
    arrange(desc(Count)) %>%
    slice_head(n = 15) %>%
    pull(Tissue)
top_tissues

[1] "Testis"                               
 [2] "Brain_Cerebellum"                     
 [3] "Brain_Cerebellar_Hemisphere"          
 [4] "Skin_Not_Sun_Exposed_Suprapubic"      
 [5] "Skin_Sun_Exposed_Lower_leg"           
 [6] "Brain_Caudate_basal_ganglia"          
 [7] "Pancreas_Islets"                      
 [8] "Brain_Hypothalamus"                   
 [9] "Brain_Nucleus_accumbens_basal_ganglia"
[10] "Brain_Putamen_basal_ganglia"          
[11] "Cells_Cultured_fibroblasts"           
[12] "Cervix_Endocervix"                    
[13] "Vagina"                               
[14] "Brain_Hippocampus"                    
[15] "Cervix_Ectocervix"

In [53]:
# manually select tissues to get more diversity
sel_tissues <- c('Testis','Brain_Cerebellum',
                'Skin_Not_Sun_Exposed_Suprapubic','Pancreas_Islets',
                'Cells_Cultured_fibroblasts','Cervix_Endocervix','Vagina')
sel_tissues_genes <- selected_genes %>% filter(Tissue %in% sel_tissues)
sel_tissues_genes %>% 
    group_by(Tissue) %>%
    summarise(Count = n()) %>%
    arrange(desc(Count))

Tissue,Count
<chr>,<int>
Testis,10
Brain_Cerebellum,8
Skin_Not_Sun_Exposed_Suprapubic,7
Pancreas_Islets,6
Cells_Cultured_fibroblasts,5
Cervix_Endocervix,5
Vagina,5


Let's hope we have enough genes to compare

In [55]:
# Get the tissue-specific genes and save them for GO analysis
# Save them in a list to later use
tissue_specific_genes <- list()
for(tissue in sel_tissues){
    tissue_genes <- sel_tissues_genes %>% 
        filter(Tissue == tissue) %>% 
        pull(Gene)

    # Add them to the list
    tissue_specific_genes[[tissue]] <- tissue_genes

    # Save them to a file
    write.table(tissue_genes, file = paste0(out_dir, "GTEx_tissue_specific_TF_genes_", tissue, "_", date, ".tsv"), 
                quote = FALSE, row.names = FALSE, col.names = FALSE, sep = "\t")
}

Let's also generate tissue-specific backgrounds as those GTEx genes with no zero expression

In [ ]:
# For each tissue, get the genes with no-zero expression in the selected tissues
for(tissue in sel_tissues){
    
    tissue_data <- gtex_data %>% dplyr::select(!!sym(tissue))
    
    # Get genes with no-zero expression in the tissue samples
    tissue_genes_no_zero <- tissue_data %>%
        filter(!!sym(tissue) > 0)
    tissue_genes_no_zero <-rownames(tissue_genes_no_zero)
    
    # Remove the tissue-specific genes from the TissueEnrich output
    current_ts_genes <- tissue_specific_genes[[tissue]]
    tissue_genes_no_zero <- tissue_genes_no_zero[!tissue_genes_no_zero %in% current_ts_genes]
    
    cat("Number of genes with no-zero expression in ", tissue, ": ", length(tissue_genes_no_zero), "\n")

    # Save the background genes to a file
    write.table(tissue_genes_no_zero, file = paste0(out_dir, "GTEx_background_genes_", tissue, "_", date, ".tsv"), 
                quote = FALSE, row.names = FALSE, col.names = FALSE, sep = "\t")
}

Number of genes with no-zero expression in  Testis :  1963 
Number of genes with no-zero expression in  Brain_Cerebellum :  1676 
Number of genes with no-zero expression in  Skin_Not_Sun_Exposed_Suprapubic :  1674 
Number of genes with no-zero expression in  Pancreas_Islets :  1606 
Number of genes with no-zero expression in  Cells_Cultured_fibroblasts :  1516 
Number of genes with no-zero expression in  Cervix_Endocervix :  1691 
Number of genes with no-zero expression in  Vagina :  1660 
